In [ ]:
# ============================================================
# YOLOv8 n/s/m - 5 RUNS PER MODEL - SEEDS 0,1,2,3,4
# REPRODUCIBLE EXPERIMENTAL PIPELINE
#
# MAIN METRICS:
#   Precision, Recall, mAP50, mAP50-95
#   -> extracted from SAME best epoch selected by max mAP50-95
#
# EFFICIENCY:
#   GPU inference latency + FPS (batch=1)
#   Parameters, GFLOPs, checkpoint size
#
# OUTPUT:
#   Excel + CSV backups + checkpoints + confusion matrices
#   + 300-DPI figures + ZIP backup
# ============================================================


# ============================================================
# 0. INSTALL
# ============================================================

!pip install -q ultralytics roboflow openpyxl thop

import os
import gc
import time
import shutil
import random
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from ultralytics import YOLO
import ultralytics


# ============================================================
# 1. ROBOFLOW DATASET
# ============================================================

from roboflow import Roboflow

# ------------------------------------------------------------
# IMPORTANT:
# Keep EXACTLY the same Roboflow project/version/split
# used in the original experiment.
# ------------------------------------------------------------

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace(os.environ["ROBOFLOW_WORKSPACE"]).project(os.environ["ROBOFLOW_PROJECT"])
version = project.version(int(os.environ.get("ROBOFLOW_VERSION", "3")))
dataset = version.download("yolov8")

DATA_YAML = Path(dataset.location) / "data.yaml"

# ============================================================
# 2. VERIFY DATASET BEFORE TRAINING
# ============================================================

print("=" * 80)
print("DATASET VERIFICATION")
print("=" * 80)

print("Dataset location :", dataset.location)
print("DATA_YAML        :", DATA_YAML)
print("Exists           :", DATA_YAML.exists())

assert DATA_YAML.exists(), (
    f"\nERROR: data.yaml not found:\n{DATA_YAML}"
)

print("\nCONTENT OF data.yaml")
print("-" * 80)

with open(DATA_YAML, "r") as f:
    yaml_content = f.read()

print(yaml_content)

print("-" * 80)


# ============================================================
# 3. EXPERIMENT CONFIGURATION
# ============================================================

OUTPUT_ROOT = Path("/content/YOLOv8_RERUN")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


MODELS = {
    "YOLOv8n": "yolov8n.pt",
    "YOLOv8s": "yolov8s.pt",
    "YOLOv8m": "yolov8m.pt",
}


# ------------------------------------------------------------
# AUTOMATIC SEEDS
# ------------------------------------------------------------

SEEDS = [0, 1, 2, 3, 4]


# ------------------------------------------------------------
# TRAINING CONFIGURATION
# ------------------------------------------------------------

EPOCHS = 50
IMGSZ = 640
BATCH = 16

OPTIMIZER = "SGD"
LR0 = 0.01


# ------------------------------------------------------------
# AUGMENTATION
# Explicitly fixed to avoid dependence on future defaults
# ------------------------------------------------------------

AUGMENTATION = {

    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,

    "translate": 0.1,
    "scale": 0.5,

    "fliplr": 0.5,

    "mosaic": 1.0,

    # Disabled
    "flipud": 0.0,
    "degrees": 0.0,
    "shear": 0.0,
    "perspective": 0.0,
    "mixup": 0.0,
}


# ------------------------------------------------------------
# INFERENCE BENCHMARK
#
# IMPORTANT:
# This is the GPU benchmark of the current Colab GPU.
# CPU benchmark / Pareto analysis will be performed separately.
# ------------------------------------------------------------

WARMUP_RUNS = 20
INFERENCE_RUNS = 100
INFERENCE_BATCH = 1


# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

DEVICE = 0 if torch.cuda.is_available() else "cpu"

GPU_NAME = (
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)


# ============================================================
# 4. PRINT COMPLETE CONFIGURATION
# ============================================================

print("\n")
print("=" * 80)
print("EXPERIMENT CONFIGURATION")
print("=" * 80)

print("Ultralytics version :", ultralytics.__version__)
print("PyTorch version     :", torch.__version__)
print("Python              :", platform.python_version())

print("Device              :", DEVICE)
print("GPU                 :", GPU_NAME)

print("\nDataset:")
print(DATA_YAML)

print("\nModels:")
for model_name, weight in MODELS.items():
    print(f"  {model_name}: {weight}")

print("\nSeeds:")
print(SEEDS)

print("\nTraining:")
print("Epochs     :", EPOCHS)
print("Image size :", IMGSZ)
print("Batch      :", BATCH)
print("Optimizer  :", OPTIMIZER)
print("LR0        :", LR0)

print("\nAugmentation:")
for key, value in AUGMENTATION.items():
    print(f"{key:15s}: {value}")

print("\nInference benchmark:")
print("Batch      :", INFERENCE_BATCH)
print("Warm-up    :", WARMUP_RUNS)
print("Runs       :", INFERENCE_RUNS)

print("=" * 80)


# ============================================================
# 5. AUXILIARY FUNCTIONS
# ============================================================

def clean_columns(df):

    df.columns = [
        str(c).strip()
        for c in df.columns
    ]

    return df


# ------------------------------------------------------------
# BEST EPOCH
# ------------------------------------------------------------

def get_best_epoch_metrics(results_csv):
    """
    Select the epoch that maximizes mAP50-95.

    Precision, Recall, mAP50 and mAP50-95 are ALL extracted
    from this SAME epoch.

    This preserves methodological consistency with the
    reconstructed YOLOv5 procedure.
    """

    df = pd.read_csv(results_csv)

    df = clean_columns(df)


    map95_col = "metrics/mAP50-95(B)"
    map50_col = "metrics/mAP50(B)"

    precision_col = "metrics/precision(B)"
    recall_col = "metrics/recall(B)"


    required = [

        map95_col,
        map50_col,

        precision_col,
        recall_col
    ]


    missing = [

        col for col in required
        if col not in df.columns
    ]


    if missing:

        raise ValueError(

            f"\nMissing columns in:\n{results_csv}\n\n"
            f"Missing:\n{missing}\n\n"
            f"Available:\n{list(df.columns)}"
        )


    best_idx = df[map95_col].idxmax()

    best = df.loc[best_idx]


    # Preserve both the value recorded by Ultralytics
    # and human-readable row position.

    if "epoch" in df.columns:

        epoch_raw = int(best["epoch"])

    else:

        epoch_raw = int(best_idx)


    epoch_row = int(best_idx) + 1


    return {

        "Best_Epoch_Raw":
            epoch_raw,

        "Best_Epoch_Row":
            epoch_row,

        "Precision":
            float(best[precision_col]),

        "Recall":
            float(best[recall_col]),

        "mAP50":
            float(best[map50_col]),

        "mAP50_95":
            float(best[map95_col]),
    }


# ------------------------------------------------------------
# CHECKPOINT SIZE
# ------------------------------------------------------------

def file_size_mb(path):

    return (
        os.path.getsize(path)
        / (1024 ** 2)
    )


# ------------------------------------------------------------
# PARAMETERS
# ------------------------------------------------------------

def count_parameters(model):

    return sum(

        p.numel()

        for p in model.parameters()
    )


# ------------------------------------------------------------
# GFLOPs
# ------------------------------------------------------------

def get_gflops(model):

    try:

        info = model.info(
            verbose=False
        )

        # Ultralytics commonly returns:
        # layers, parameters, gradients, GFLOPs

        if isinstance(info, tuple):

            if len(info) >= 4:

                return float(info[-1])


        return np.nan


    except Exception as e:

        print(
            "WARNING: GFLOPs could not "
            "be obtained automatically:",
            e
        )

        return np.nan


# ------------------------------------------------------------
# GPU INFERENCE BENCHMARK
# ------------------------------------------------------------

def measure_inference(
    model,
    imgsz=640,
    warmup=20,
    repetitions=100
):

    """
    Pure forward-pass benchmark.

    Batch = 1
    Synthetic tensor
    Fixed image size

    Disk I/O is excluded.

    NOTE:
    This measures GPU forward-pass latency.
    CPU deployment benchmark will be performed separately.
    """

    model.model.eval()

    device = next(
        model.model.parameters()
    ).device


    dummy = torch.zeros(

        (
            1,
            3,
            imgsz,
            imgsz
        ),

        device=device
    )


    # ------------------------------
    # WARM-UP
    # ------------------------------

    with torch.no_grad():

        for _ in range(warmup):

            _ = model.model(dummy)


    if torch.cuda.is_available():

        torch.cuda.synchronize()


    times = []


    # ------------------------------
    # MEASUREMENT
    # ------------------------------

    with torch.no_grad():

        for _ in range(repetitions):

            if torch.cuda.is_available():

                torch.cuda.synchronize()


            start = time.perf_counter()


            _ = model.model(dummy)


            if torch.cuda.is_available():

                torch.cuda.synchronize()


            end = time.perf_counter()


            times.append(

                (end - start) * 1000
            )


    times = np.array(times)


    mean_ms = float(
        np.mean(times)
    )

    sd_ms = float(
        np.std(times, ddof=1)
    )

    median_ms = float(
        np.median(times)
    )


    fps = (
        1000.0 / mean_ms
    )


    return {

        "Inference_ms_Mean":
            mean_ms,

        "Inference_ms_SD":
            sd_ms,

        "Inference_ms_Median":
            median_ms,

        "FPS":
            fps
    }


# ============================================================
# 6. STORAGE
# ============================================================

all_results = []

checkpoint_records = []


# ============================================================
# 7. TRAINING LOOP
# ============================================================

TOTAL_RUNS = (
    len(MODELS)
    * len(SEEDS)
)

current_run = 0


for model_name, weights in MODELS.items():


    print("\n")
    print("#" * 80)
    print(f"STARTING MODEL: {model_name}")
    print("#" * 80)


    model_root = (

        OUTPUT_ROOT
        / model_name
    )


    model_root.mkdir(

        parents=True,
        exist_ok=True
    )


    for seed in SEEDS:


        current_run += 1


        print("\n")
        print("=" * 80)

        print(
            f"RUN {current_run}/{TOTAL_RUNS}"
        )

        print(
            f"{model_name} | SEED {seed}"
        )

        print("=" * 80)


        # ----------------------------------------------------
        # Explicit reproducibility
        # ----------------------------------------------------

        random.seed(seed)

        np.random.seed(seed)

        torch.manual_seed(seed)


        if torch.cuda.is_available():

            torch.cuda.manual_seed_all(seed)


        # ----------------------------------------------------
        # RUN NAME
        # ----------------------------------------------------

        run_name = (
            f"{model_name}_seed_{seed}"
        )


        # ----------------------------------------------------
        # LOAD PRETRAINED MODEL
        # ----------------------------------------------------

        model = YOLO(weights)


        # ----------------------------------------------------
        # TRAIN
        # ----------------------------------------------------

        model.train(

            data=str(DATA_YAML),

            epochs=EPOCHS,

            imgsz=IMGSZ,

            batch=BATCH,

            optimizer=OPTIMIZER,

            lr0=LR0,


            # ------------------------
            # REPRODUCIBILITY
            # ------------------------

            seed=seed,

            deterministic=True,


            # ------------------------
            # AUGMENTATION
            # ------------------------

            hsv_h=AUGMENTATION["hsv_h"],

            hsv_s=AUGMENTATION["hsv_s"],

            hsv_v=AUGMENTATION["hsv_v"],


            translate=
                AUGMENTATION["translate"],

            scale=
                AUGMENTATION["scale"],


            fliplr=
                AUGMENTATION["fliplr"],


            mosaic=
                AUGMENTATION["mosaic"],


            flipud=
                AUGMENTATION["flipud"],

            degrees=
                AUGMENTATION["degrees"],

            shear=
                AUGMENTATION["shear"],

            perspective=
                AUGMENTATION["perspective"],

            mixup=
                AUGMENTATION["mixup"],


            # ------------------------
            # HARDWARE
            # ------------------------

            device=DEVICE,


            # ------------------------
            # OUTPUT
            # ------------------------

            project=str(model_root),

            name=run_name,

            exist_ok=False,

            plots=True,

            save=True,

            verbose=True
        )


        # ====================================================
        # 8. LOCATE OUTPUT FILES
        # ====================================================

        run_dir = Path(
            model.trainer.save_dir
        )


        results_csv = (

            run_dir
            / "results.csv"
        )


        best_pt = (

            run_dir
            / "weights"
            / "best.pt"
        )


        last_pt = (

            run_dir
            / "weights"
            / "last.pt"
        )


        print("\nRun directory:")
        print(run_dir)


        assert results_csv.exists(), (
            f"Missing results.csv: {results_csv}"
        )


        assert best_pt.exists(), (
            f"Missing best.pt: {best_pt}"
        )


        # ====================================================
        # 9. BEST EPOCH METRICS
        # ====================================================

        metrics = get_best_epoch_metrics(
            results_csv
        )


        print("\nBEST EPOCH")

        print(
            "Raw epoch value :",
            metrics["Best_Epoch_Raw"]
        )

        print(
            "CSV row         :",
            metrics["Best_Epoch_Row"]
        )

        print(
            "Precision       :",
            metrics["Precision"]
        )

        print(
            "Recall          :",
            metrics["Recall"]
        )

        print(
            "mAP50           :",
            metrics["mAP50"]
        )

        print(
            "mAP50-95        :",
            metrics["mAP50_95"]
        )


        # ====================================================
        # 10. LOAD BEST CHECKPOINT
        # ====================================================

        best_model = YOLO(
            str(best_pt)
        )


        # ====================================================
        # 11. PARAMETERS
        # ====================================================

        params = count_parameters(
            best_model.model
        )


        # ====================================================
        # 12. GFLOPs
        # ====================================================

        gflops = get_gflops(
            best_model
        )


        # ====================================================
        # 13. CHECKPOINT SIZE
        # ====================================================

        size_mb = file_size_mb(
            best_pt
        )


        # ====================================================
        # 14. GPU INFERENCE / FPS
        # ====================================================

        inference = measure_inference(

            best_model,

            imgsz=IMGSZ,

            warmup=WARMUP_RUNS,

            repetitions=INFERENCE_RUNS
        )


        print("\nGPU INFERENCE BENCHMARK")

        print(
            "Mean latency (ms):",
            inference["Inference_ms_Mean"]
        )

        print(
            "SD latency (ms):",
            inference["Inference_ms_SD"]
        )

        print(
            "Median latency (ms):",
            inference["Inference_ms_Median"]
        )

        print(
            "FPS:",
            inference["FPS"]
        )


        # ====================================================
        # 15. STORE RESULT
        # ====================================================

        row = {

            "Model":
                model_name,

            "Seed":
                seed,

            "Best_Epoch_Raw":
                metrics["Best_Epoch_Raw"],

            "Best_Epoch_Row":
                metrics["Best_Epoch_Row"],


            "Precision":
                metrics["Precision"],

            "Recall":
                metrics["Recall"],

            "mAP50":
                metrics["mAP50"],

            "mAP50_95":
                metrics["mAP50_95"],


            "GPU_Inference_ms_Mean":
                inference[
                    "Inference_ms_Mean"
                ],

            "GPU_Inference_ms_SD":
                inference[
                    "Inference_ms_SD"
                ],

            "GPU_Inference_ms_Median":
                inference[
                    "Inference_ms_Median"
                ],

            "GPU_FPS":
                inference["FPS"],


            "Parameters":
                params,

            "GFLOPs":
                gflops,

            "Size_MB":
                size_mb,


            "GPU":
                GPU_NAME,

            "Ultralytics_Version":
                ultralytics.__version__,


            "Best_Checkpoint":
                str(best_pt),

            "Last_Checkpoint":
                str(last_pt),

            "Results_CSV":
                str(results_csv),

            "Run_Directory":
                str(run_dir),
        }


        all_results.append(row)


        checkpoint_records.append({

            "Model":
                model_name,

            "Seed":
                seed,

            "Best_pt":
                str(best_pt),

            "Last_pt":
                str(last_pt),

            "Results_csv":
                str(results_csv),

            "Run_directory":
                str(run_dir),
        })


        # ====================================================
        # 16. BACKUP AFTER EVERY RUN
        # ====================================================

        backup_df = pd.DataFrame(
            all_results
        )


        backup_df.to_csv(

            OUTPUT_ROOT
            / "RUN_RESULTS_BACKUP.csv",

            index=False
        )


        backup_df.to_excel(

            OUTPUT_ROOT
            / "RUN_RESULTS_BACKUP.xlsx",

            index=False
        )


        print(
            "\nBackup saved successfully."
        )


        # ====================================================
        # 17. CLEAN GPU MEMORY
        # ====================================================

        del model
        del best_model


        gc.collect()


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


# ============================================================
# 18. INDIVIDUAL RUN RESULTS
# ============================================================

df_runs = pd.DataFrame(
    all_results
)


print("\n")
print("=" * 80)
print("INDIVIDUAL RUN RESULTS")
print("=" * 80)

display(df_runs)


# ============================================================
# 19. MEAN ± SAMPLE SD
# ============================================================

metrics_to_summarize = [

    "Precision",

    "Recall",

    "mAP50",

    "mAP50_95",

    "GPU_Inference_ms_Mean",

    "GPU_FPS",
]


summary_rows = []


for model_name in MODELS.keys():


    subset = df_runs[

        df_runs["Model"]
        == model_name
    ]


    row = {

        "Model":
            model_name
    }


    for metric in metrics_to_summarize:


        row[
            f"{metric}_Mean"
        ] = subset[
            metric
        ].mean()


        # Sample SD: n - 1
        row[
            f"{metric}_SD"
        ] = subset[
            metric
        ].std(ddof=1)


    # Architecture information

    row["Parameters"] = (
        subset["Parameters"].iloc[0]
    )


    row["GFLOPs"] = (
        subset["GFLOPs"].mean()
    )


    row["Size_MB_Mean"] = (
        subset["Size_MB"].mean()
    )


    row["Size_MB_SD"] = (
        subset["Size_MB"].std(
            ddof=1
        )
    )


    summary_rows.append(row)


df_summary = pd.DataFrame(
    summary_rows
)


print("\n")
print("=" * 80)
print("MEAN ± SAMPLE SD")
print("=" * 80)

display(df_summary)


# ============================================================
# 20. PAPER-FORMATTED TABLE
# ============================================================

paper_rows = []


for _, row in df_summary.iterrows():


    paper_rows.append({

        "Model":

            row["Model"],


        "Precision":

            (
                f'{row["Precision_Mean"]:.3f}'
                f' ± '
                f'{row["Precision_SD"]:.3f}'
            ),


        "Recall":

            (
                f'{row["Recall_Mean"]:.3f}'
                f' ± '
                f'{row["Recall_SD"]:.3f}'
            ),


        "mAP50":

            (
                f'{row["mAP50_Mean"]:.3f}'
                f' ± '
                f'{row["mAP50_SD"]:.3f}'
            ),


        "mAP50-95":

            (
                f'{row["mAP50_95_Mean"]:.3f}'
                f' ± '
                f'{row["mAP50_95_SD"]:.3f}'
            ),


        "GPU Latency (ms)":

            (
                f'{row["GPU_Inference_ms_Mean_Mean"]:.2f}'
                f' ± '
                f'{row["GPU_Inference_ms_Mean_SD"]:.2f}'
            ),


        "GPU FPS":

            (
                f'{row["GPU_FPS_Mean"]:.2f}'
                f' ± '
                f'{row["GPU_FPS_SD"]:.2f}'
            ),


        "Parameters":

            int(
                row["Parameters"]
            ),


        "GFLOPs":

            (
                round(
                    row["GFLOPs"],
                    2
                )

                if not pd.isna(
                    row["GFLOPs"]
                )

                else np.nan
            ),


        "Size (MB)":

            (
                f'{row["Size_MB_Mean"]:.2f}'
                f' ± '
                f'{row["Size_MB_SD"]:.2f}'
            ),
    })


df_paper = pd.DataFrame(
    paper_rows
)


print("\n")
print("=" * 80)
print("PAPER TABLE")
print("=" * 80)

display(df_paper)


# ============================================================
# 21. CONFIGURATION TABLE
# ============================================================

configuration = pd.DataFrame({

    "Parameter": [

        "Models",

        "Runs per model",

        "Seeds",

        "Epochs",

        "Image size",

        "Training batch",

        "Optimizer",

        "Initial learning rate",

        "hsv_h",

        "hsv_s",

        "hsv_v",

        "Translation",

        "Scaling",

        "Horizontal flip",

        "Mosaic",

        "Vertical flip",

        "Rotation",

        "Shear",

        "Perspective",

        "MixUp",

        "Deterministic",

        "Inference batch",

        "Inference warm-up",

        "Inference repetitions",

        "Ultralytics version",

        "PyTorch version",

        "GPU",
    ],


    "Value": [

        "YOLOv8n, YOLOv8s, YOLOv8m",

        5,

        "0, 1, 2, 3, 4",

        EPOCHS,

        IMGSZ,

        BATCH,

        OPTIMIZER,

        LR0,

        AUGMENTATION["hsv_h"],

        AUGMENTATION["hsv_s"],

        AUGMENTATION["hsv_v"],

        AUGMENTATION["translate"],

        AUGMENTATION["scale"],

        AUGMENTATION["fliplr"],

        AUGMENTATION["mosaic"],

        AUGMENTATION["flipud"],

        AUGMENTATION["degrees"],

        AUGMENTATION["shear"],

        AUGMENTATION["perspective"],

        AUGMENTATION["mixup"],

        True,

        INFERENCE_BATCH,

        WARMUP_RUNS,

        INFERENCE_RUNS,

        ultralytics.__version__,

        torch.__version__,

        GPU_NAME,
    ]
})


# ============================================================
# 22. CHECKPOINT TABLE
# ============================================================

df_checkpoints = pd.DataFrame(
    checkpoint_records
)


# ============================================================
# 23. FIGURE DIRECTORY
# ============================================================

FIG_DIR = (

    OUTPUT_ROOT
    / "figures"
)

FIG_DIR.mkdir(
    exist_ok=True
)


# ============================================================
# 24. FUNCTION FOR 300-DPI BAR CHART
# ============================================================

def bar_chart(
    means,
    errors,
    ylabel,
    filename,
    ylim=None
):

    fig, ax = plt.subplots(
        figsize=(8, 5)
    )


    x = np.arange(
        len(df_summary)
    )


    ax.bar(

        x,

        means,

        yerr=errors,

        capsize=5
    )


    ax.set_xticks(x)


    ax.set_xticklabels(
        df_summary["Model"]
    )


    ax.set_xlabel(
        "Model"
    )


    ax.set_ylabel(
        ylabel
    )


    if ylim is not None:

        ax.set_ylim(
            ylim
        )


    ax.grid(

        axis="y",

        linestyle="--",

        alpha=0.4
    )


    fig.tight_layout()


    fig.savefig(

        FIG_DIR / filename,

        dpi=300,

        bbox_inches="tight"
    )


    plt.show()


# ============================================================
# 25. ACCURACY FIGURES - 300 DPI
# ============================================================

bar_chart(

    df_summary[
        "mAP50_95_Mean"
    ],

    df_summary[
        "mAP50_95_SD"
    ],

    "mAP@0.5:0.95",

    "mAP50_95_mean_SD.png",

    (0, 1)
)


bar_chart(

    df_summary[
        "mAP50_Mean"
    ],

    df_summary[
        "mAP50_SD"
    ],

    "mAP@0.5",

    "mAP50_mean_SD.png",

    (0, 1)
)


bar_chart(

    df_summary[
        "Precision_Mean"
    ],

    df_summary[
        "Precision_SD"
    ],

    "Precision",

    "Precision_mean_SD.png",

    (0, 1)
)


bar_chart(

    df_summary[
        "Recall_Mean"
    ],

    df_summary[
        "Recall_SD"
    ],

    "Recall",

    "Recall_mean_SD.png",

    (0, 1)
)


# ============================================================
# 26. GPU FPS FIGURE - 300 DPI
# ============================================================

bar_chart(

    df_summary[
        "GPU_FPS_Mean"
    ],

    df_summary[
        "GPU_FPS_SD"
    ],

    "GPU FPS (batch = 1)",

    "GPU_FPS_mean_SD.png"
)


# ============================================================
# 27. GPU LATENCY FIGURE - 300 DPI
# ============================================================

bar_chart(

    df_summary[
        "GPU_Inference_ms_Mean_Mean"
    ],

    df_summary[
        "GPU_Inference_ms_Mean_SD"
    ],

    "GPU inference latency (ms)",

    "GPU_latency_mean_SD.png"
)


# ============================================================
# 28. PRELIMINARY GPU ACCURACY-EFFICIENCY FIGURE
#
# NOTE:
# NOT the final CPU Pareto figure requested by E1.
# ============================================================

fig, ax = plt.subplots(
    figsize=(8, 5)
)


ax.scatter(

    df_summary[
        "GPU_Inference_ms_Mean_Mean"
    ],

    df_summary[
        "mAP50_95_Mean"
    ],

    s=100
)


for _, row in df_summary.iterrows():


    ax.annotate(

        row["Model"],

        (

            row[
                "GPU_Inference_ms_Mean_Mean"
            ],

            row[
                "mAP50_95_Mean"
            ]
        ),

        xytext=(5, 5),

        textcoords="offset points"
    )


ax.set_xlabel(
    "GPU inference latency (ms)"
)


ax.set_ylabel(
    "mAP@0.5:0.95"
)


ax.grid(
    linestyle="--",
    alpha=0.4
)


fig.tight_layout()


fig.savefig(

    FIG_DIR
    / "GPU_accuracy_efficiency_PRELIMINARY.png",

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# ============================================================
# 29. NORMALIZED CONFUSION MATRICES
# ============================================================

CM_DIR = (

    OUTPUT_ROOT
    / "normalized_confusion_matrices"
)


CM_DIR.mkdir(
    exist_ok=True
)


print("\n")
print("=" * 80)
print("GENERATING NORMALIZED CONFUSION MATRICES")
print("=" * 80)


for _, row in df_checkpoints.iterrows():


    model_name = row["Model"]

    seed = int(
        row["Seed"]
    )

    best_pt = row["Best_pt"]


    print(
        f"\n{model_name} | seed={seed}"
    )


    model = YOLO(
        best_pt
    )


    cm_name = (

        f"{model_name}"
        f"_seed_{seed}"
    )


    model.val(

        data=str(DATA_YAML),

        imgsz=IMGSZ,

        batch=1,

        device=DEVICE,

        plots=True,

        project=str(CM_DIR),

        name=cm_name,

        exist_ok=True,

        verbose=False
    )


    del model


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 30. EXCEL EXPORT
# ============================================================

EXCEL_PATH = (

    OUTPUT_ROOT
    / "YOLOv8_FINAL_RESULTS.xlsx"
)


with pd.ExcelWriter(

    EXCEL_PATH,

    engine="openpyxl"

) as writer:


    df_runs.to_excel(

        writer,

        sheet_name="RUN_RESULTS",

        index=False
    )


    df_summary.to_excel(

        writer,

        sheet_name="SUMMARY",

        index=False
    )


    df_paper.to_excel(

        writer,

        sheet_name="PAPER_TABLE",

        index=False
    )


    configuration.to_excel(

        writer,

        sheet_name="CONFIGURATION",

        index=False
    )


    df_checkpoints.to_excel(

        writer,

        sheet_name="CHECKPOINTS",

        index=False
    )


# ============================================================
# 31. FORMAT EXCEL
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment


wb = load_workbook(
    EXCEL_PATH
)


for ws in wb.worksheets:


    ws.freeze_panes = "A2"


    for cell in ws[1]:

        cell.font = Font(
            bold=True
        )

        cell.alignment = Alignment(

            horizontal="center",

            vertical="center"
        )


    for column_cells in ws.columns:


        max_length = 0


        column_letter = (

            column_cells[0]
            .column_letter
        )


        for cell in column_cells:


            try:

                length = len(
                    str(cell.value)
                )


                max_length = max(

                    max_length,

                    length
                )


            except:

                pass


        ws.column_dimensions[
            column_letter
        ].width = min(

            max_length + 2,

            45
        )


wb.save(
    EXCEL_PATH
)


# ============================================================
# 32. SAVE ADDITIONAL CSV FILES
# ============================================================

df_runs.to_csv(

    OUTPUT_ROOT
    / "FINAL_RUN_RESULTS.csv",

    index=False
)


df_summary.to_csv(

    OUTPUT_ROOT
    / "FINAL_SUMMARY.csv",

    index=False
)


df_paper.to_csv(

    OUTPUT_ROOT
    / "FINAL_PAPER_TABLE.csv",

    index=False
)


# ============================================================
# 33. FINAL ZIP BACKUP
# ============================================================

ZIP_PATH = (
    "/content/YOLOv8_RERUN_COMPLETE"
)


shutil.make_archive(

    ZIP_PATH,

    "zip",

    OUTPUT_ROOT
)


# ============================================================
# 34. FINAL REPORT
# ============================================================

print("\n")
print("=" * 80)
print("EXPERIMENT FINISHED SUCCESSFULLY")
print("=" * 80)


print("\nExcel:")
print(EXCEL_PATH)


print("\nFigures:")
print(FIG_DIR)


print("\nConfusion matrices:")
print(CM_DIR)


print("\nComplete ZIP:")
print(
    ZIP_PATH + ".zip"
)


print("\nFINAL PAPER TABLE:")

display(
    df_paper
)


print("\n")
print("=" * 80)

print(
    "IMPORTANT: GPU latency/FPS are preliminary "
    "hardware-specific measurements."
)

print(
    "The final CPU latency + Pareto analysis for E1/R3 "
    "will be performed separately using all six models."
)

print("=" * 80)

In [ ]:
from pathlib import Path
import zipfile

ROOT = Path("/content/YOLOv8_RERUN")
ZIP_PATH = Path("/content/YOLOv8_BEST_CHECKPOINTS.zip")

best_files = sorted(ROOT.rglob("best.pt"))

print(f"Encontrados: {len(best_files)} best.pt")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for pt in best_files:
        # Conserva modelo + seed para evitar tener 15 archivos llamados best.pt
        relative = pt.relative_to(ROOT)
        z.write(pt, arcname=str(relative))

print(f"\nZIP creado: {ZIP_PATH}")

In [ ]:
from google.colab import files
files.download("/content/YOLOv8_BEST_CHECKPOINTS.zip")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Cambia el nombre si tu archivo se llama diferente
font_path = "/content/times.ttf"

# Registrar la fuente en Matplotlib
fm.fontManager.addfont(font_path)

# Obtener el nombre interno real de la fuente
font_name = fm.FontProperties(fname=font_path).get_name()

print("Fuente detectada:", font_name)

# Configurar toda la figura con esa fuente
plt.rcParams["font.family"] = font_name
plt.rcParams["font.size"] = 8

In [ ]:
import matplotlib.pyplot as plt

# =========================
# Data
# =========================
training_images = [268, 536, 804, 1072, 1340]
map_50_95 = [0.483, 0.533, 0.574, 0.593, 0.599]

# =========================
# IJACSA figure settings
# =========================
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8
})

# Single-column figure size
fig, ax = plt.subplots(figsize=(3.5, 2.6))

# =========================
# Plot
# =========================
ax.plot(
    training_images,
    map_50_95,
    marker="o",
    linewidth=1.2,
    markersize=4
)

# Value labels
# Value labels
for i, (x, y) in enumerate(zip(training_images, map_50_95)):
    if i == 0:
        xytext = (3, 6)   # Move 0.483 slightly up/right
    else:
        xytext = (0, 5)

    ax.annotate(
        f"{y:.3f}",
        (x, y),
        xytext=xytext,
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=8
    )

# =========================
# Axes
# =========================
ax.set_xlabel("Number of training images")
ax.set_ylabel("mAP@0.5:0.95")

ax.set_xticks(training_images)
ax.set_ylim(0.48, 0.61)

ax.grid(
    True,
    linestyle="--",
    linewidth=0.5,
    alpha=0.5
)

# Remove unnecessary top/right borders
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# =========================
# Save publication-quality
# =========================
plt.savefig(
    "Fig1_YOLOv8n_dataset_size.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "Fig1_YOLOv8n_dataset_size.pdf",
    bbox_inches="tight"
)

plt.show()